# Cluster selection — working notebook

Picking which time-cluster carries the hard-scatter vertex time. Superseded exploration
lives in `training_archive.ipynb`; **everything here is meant to be run.**

**Where things stand.** Best selector is a Deep Sets model trained end-to-end on the
*selection* objective — Z+jets **68.1%** core fraction vs TRKPTZ 62.1%, best-in-class on
all three topologies. Four other axes were tried and are exhausted: cluster features,
per-track tagger features, model capacity, and pooling form. The one change that ever
worked was changing the **objective**.

**Two ceilings to keep in view.** A cluster within 60 ps exists in **92.2%** of Z+jets
events (the oracle). Selecting by `Σ pT·truth_is_hs` — a *perfect* per-track HS tagger
with the simplest pooling — reaches only **80.2%**. So the residual splits into
identification (68→80) and *time quality* (80→92), and the second part is not reachable
by any per-track HS weighting.

**What is new in this notebook.** Two things:

1. `SELECTION = "loose"` — the `--vbs-deta=0` export, which drops the VBS topology cut.
   That cut removed 87% of Z+jets events surviving the lepton selection and may have been
   *manufacturing* the forward-jet-pileup pathology rather than exposing it. **The first
   number to look at is TRKPTZ on Z+jets in §3**: meaningfully above 62.1% means the old
   selection was the problem.
2. An **auxiliary per-track head** on the Deep Sets model (§7), scanned over λ including
   λ = 0 so the control is internal.

> ⚠️ The loose export is a strict **superset** of the tight one with identical
> `event_num`s, so the two must never be loaded together. §2 asserts uniqueness.

## 1. Setup

In [ ]:
import glob, os, warnings
import numpy as np, pandas as pd, uproot
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

# Optional backends from the vtx venv. Guarded so the notebook still runs on the
# stock kernel, just without xgboost/lightgbm sections.
try:
    import xgboost as xgb
    HAVE_XGB = True
except ImportError:
    HAVE_XGB = False
try:
    import lightgbm as lgb
    HAVE_LGB = True
except ImportError:
    HAVE_LGB = False
try:
    import torch
    HAVE_TORCH = True
    GPU = torch.cuda.is_available()
except ImportError:
    HAVE_TORCH, GPU = False, False
print(f"xgboost: {xgb.__version__ if HAVE_XGB else 'MISSING'}   "
      f"lightgbm: {lgb.__version__ if HAVE_LGB else 'MISSING'}   "
      f"torch: {torch.__version__ if HAVE_TORCH else 'MISSING'}"
      + ("" if (HAVE_XGB and HAVE_LGB) else "   <- switch to the Python (vtx) kernel"))
if HAVE_TORCH:
    print(f"GPU: {torch.cuda.get_device_name(0) if GPU else 'none visible'}"
          + ("" if GPU else "   (torch may be the CPU-only build -- see setup_vtx.sh)"))
warnings.filterwarnings("ignore")

# --- where the exporter's ROOT files live -----------------------------------
# condor writes <repo>/<sample>/<sample>_training.root; the local run writes
# figs/hists/training.root. Adjust SEARCH_DIRS if you staged them elsewhere.
SEARCH_DIRS = [".", "..", "../..", "figs/hists", "../figs/hists"]
SAMPLES     = ["vbf", "zjets", "dijet"]

# "tight" = the standard selection (<s>_training.root).
# "loose" = the VBS-topology cut dropped, --vbs-deta=0 (<s>_deta0p0_training.root),
#           falling back to the tight file for any sample not re-exported.
#
# NEVER load both for the same sample. The loose selection is a strict SUPERSET
# of the tight one and the exporter writes the same sample_id and the same
# event_num, so concatenating them would duplicate every tight event under a
# key that is supposed to be unique -- silently corrupting the event-level
# split, the listwise groups and every core fraction. The guard in the load
# cell asserts uniqueness rather than trusting this comment.
SELECTION   = "loose"          # "tight" | "loose"
LOOSE_TAG   = "deta0p0"

PASS_PS   = 60.0     # the physics window (PASS_SIGMA in clustering_constants.h)
EVT       = ["sample_id", "event_num"]   # ranking group key
RANDOM    = 0

SAMPLE_NAME = {0.0: "vbf", 1.0: "zjets", 2.0: "dijet", 3.0: "local"}

def _find(basename):
    for d in SEARCH_DIRS:
        for p in glob.glob(os.path.join(d, "*", basename)) + \
                 glob.glob(os.path.join(d, basename)):
            return os.path.abspath(p)
    return None

def find_files(selection=SELECTION):
    hits = {}
    for s in SAMPLES:
        p = _find(f"{s}_{LOOSE_TAG}_training.root") if selection == "loose" else None
        tag = LOOSE_TAG
        if p is None:
            p, tag = _find(f"{s}_training.root"), "tight"
        if p is not None:
            hits[s] = p
            print(f"  found {s:6s} [{tag:7s}] -> {p}")
    p = _find("training.root")
    if p is not None and not hits:
        hits["local"] = p
        print(f"  found {'local':6s} [tight  ] -> {p}")
    return hits

print(f"selection: {SELECTION}")
FILES = find_files()
if not FILES:
    print("\\n!! No training ROOT files found. Run:  ./export_training_data --sample=<s>")

## 2. Load

In [ ]:
def load(files):
    parts = []
    for name, path in files.items():
        df = uproot.open(path)["clusters"].arrays(library="pd")
        parts.append(df)
        print(f"  {name:6s} {len(df):>8,} cluster rows   "
              f"{df.groupby(EVT).ngroups:>7,} events   ({path})")
    return pd.concat(parts, ignore_index=True)

df = load(FILES)
# Uniqueness of (sample_id, event_num) is load-bearing: it is the event key for
# the split, the listwise groups and every groupby in the notebook. It breaks if
# a tight and a loose export of the same sample are ever loaded together.
_dup = df.duplicated(EVT + ["cluster_idx"]).sum()
assert _dup == 0, (f"{_dup:,} duplicate (sample_id, event_num, cluster_idx) rows -- "
                   "are a tight and a loose export of the same sample both loaded?")
df["abs_dt"] = df["delta_t"].abs()
print(f"\ntotal {len(df):,} rows / {df.groupby(EVT).ngroups:,} events / {df.shape[1]} columns")

## 3. Labels and reference selectors

Labels come from `delta_t` (cluster time − truth HS time), which is the **target**, never
a feature. `is_best` = truth-closest cluster in the event; `within60` = inside the physics
window (multi-positive — several clusters can qualify).

No training here, so this is the cheap first look: if TRKPTZ on Z+jets has moved off
62.1%, the loose selection changed the problem.

In [ ]:
g = df.groupby(EVT, sort=False)["abs_dt"]
df["is_best"]  = (df["abs_dt"] == g.transform("min")).astype(int)
df["within60"] = (df["abs_dt"] < PASS_PS).astype(int)
# an event is RECOVERABLE if any cluster is inside the window; a "win" on an
# unrecoverable event is luck, so we report both inclusive and conditional numbers
df["recoverable"] = (g.transform("min") < PASS_PS).astype(int)

def select_metric(frame, col, higher_is_better=True):
    """Pick one cluster per event by `col`, return (core fraction, median |dt|)."""
    s = frame[col]
    s = s.fillna(-np.inf if higher_is_better else np.inf)
    idx = s.groupby([frame[c] for c in EVT], sort=False)
    idx = idx.idxmax() if higher_is_better else idx.idxmin()
    sel = frame.loc[idx.to_numpy()]
    return 100.0 * (sel["abs_dt"] < PASS_PS).mean(), sel["abs_dt"].median()

def report(frame, rows, title):
    print(f"\n{title}   ({frame.groupby(EVT).ngroups:,} events)")
    print(f"  {'selector':26s} {'core<60ps':>10s} {'median|dt|':>11s}")
    print("  " + "-" * 50)
    for name, col, hib in rows:
        cf, md_ = select_metric(frame, col, hib)
        print(f"  {name:26s} {cf:9.1f}% {md_:10.1f} ps")

BASELINES = [("oracle (truth-closest)", "abs_dt", False),
             ("TRKPTZ", "trkptz_score", True),
             ("WAVeS",  "waves_score",  True),
             ("sumpt",  "sumpt",        True)]
report(df, BASELINES, "ALL SAMPLES")
for sid, sub in df.groupby("sample_id"):
    report(sub, BASELINES, f"sample = {SAMPLE_NAME.get(sid, sid)}")

## 4. Features

In [ ]:
LABELS = ["is_best", "within60", "recoverable"]
# Ordering guard: this cell must run AFTER §3, which creates the label columns.
# If it runs first they are simply absent from df -- they cannot leak into feat,
# but the "held out" count silently drops and the guard is then passing by
# accident rather than by construction. (Observed: run 4 executed §4 at In[3]
# and §3 at In[4].)
assert all(c in df.columns for c in LABELS), \
    f"run \u00a73 first -- missing {[c for c in LABELS if c not in df.columns]}"

LEAK = [c for c in df.columns if c.startswith("truth_")] + \
       ["delta_t", "abs_dt"] + LABELS
IDS  = ["event_num", "cluster_idx", "sample_id", "weight"]

# Pre-rename aliases. The exporter emits these under the truth_ prefix as of
# e507e76, but a ROOT file written BEFORE that re-export still has the old bare
# name, which the prefix test above cannot catch -- and e507e76 also removed the
# manual exclusion, so on an old file the column silently re-enters the features.
# Listing the old names keeps the guard correct for both vintages; on a
# re-exported file they simply match nothing. Safe to delete only once every
# training file on disk is post-rename.
TRUTH_ALIASES = ["mean_nhgtd_primary", "nhgtd_primary"]
_stale = sorted(set(TRUTH_ALIASES) & set(df.columns))
if _stale:
    print("!! PRE-RENAME FILE: " + ", ".join(_stale) + " present under the old bare name.")
    print("   Excluded via TRUTH_ALIASES (the truth_ prefix test does not see them).")
    print("   Re-run util/export_training_data.cxx to emit them truth_-prefixed.\n")
LEAK += TRUTH_ALIASES

# Legitimate reco, but excluded deliberately -- see the notes above. Flip
# USE_HGTD_FEATURES to True only to quantify what the old-algorithm prior is worth.
USE_HGTD_FEATURES = False
HGTD_BLOCK = ["hgtd_time", "hgtd_valid", "hgtd_time_res", "dt_cluster_to_hgtd", "mean_nhgtd_primary"]
# (mean_nhgtd_primary is no longer listed here: the exporter now writes it as
#  truth_mean_nhgtd_primary, so the truth_ guard above catches it automatically.)
EXCLUDE_BY_DESIGN = [] if USE_HGTD_FEATURES else list(HGTD_BLOCK)

# Curated list for within-event normalization: continuous, physically meaningful,
# and plausibly comparative. (Normalizing all ~90 would triple the width for little gain.)
NORMALIZE = ["sumpt", "sumpt2", "maxpt", "n_tracks", "cluster_time_sigma",
             "time_chi2_ndf", "delta_z_resunits", "z_chi2_ndf", "mean_nhgtd",
             "trkptz_score", "waves_score", "frac_pt_in_fwdjet", "lead_pt_frac",
             "dz_to_lepton_signif", "sumpt_in_fwdjet"]

def add_event_context(frame):
    frame = frame.copy()
    gb = frame.groupby(EVT, sort=False)
    for c in NORMALIZE:
        if c not in frame.columns:
            continue
        mx = gb[c].transform("max")
        frame[f"{c}_ratio_to_max"] = np.where(np.abs(mx) > 0, frame[c] / mx, np.nan)
        frame[f"{c}_rank"] = gb[c].rank(ascending=False, method="min")
    return frame

df = add_event_context(df)

DROP = set(LEAK) | set(IDS) | set(EXCLUDE_BY_DESIGN)
feat = [c for c in df.columns if c not in DROP]
# auto-drop degenerate columns (evaluated on the full set)
degen = [c for c in feat
         if df[c].isna().all() or df[c].nunique(dropna=True) <= 1]
feat = [c for c in feat if c not in degen]

# Fail loudly rather than train on a truth column: cheap, and the two mechanisms
# above (prefix + alias) are exactly the kind of thing that rots when the schema moves.
_bad = [c for c in feat if c.startswith("truth_") or c in set(TRUTH_ALIASES)]
assert not _bad, f"TRUTH LEAKED INTO FEATURES: {_bad}"

print(f"dropped {len(degen)} degenerate columns:")
for c in degen: print("   ", c)
print(f"\nexcluded by design: {sorted(set(EXCLUDE_BY_DESIGN) & set(df.columns))}")
print(f"truth/label columns held out: {len(set(LEAK) & set(df.columns))}")
print(f"{len(feat)} features -> X")

## 5. Split

In [ ]:
rng = np.random.default_rng(RANDOM)
ev  = df[EVT].drop_duplicates().reset_index(drop=True)
ev["fold"] = rng.random(len(ev))
df2 = df.merge(ev, on=EVT, how="left")
train = df2[df2.fold <  0.7].copy()
test  = df2[df2.fold >= 0.7].copy()
print(f"train {train.groupby(EVT).ngroups:,} events / {len(train):,} rows")
print(f"test  {test.groupby(EVT).ngroups:,} events / {len(test):,} rows")

## 6. Cluster-level reference model

One GBDT (formulation B: *is this cluster inside the 60 ps window*), kept purely as the
same-sample comparator for §7 — on the tight selection this was the best cluster-level
result at 66.5% on Z+jets. Formulations A and C and the cross-topology matrix are in the
archive; they answered their questions and cost a fit each.

Early stopping validates on an **event-disjoint** slice carved from train (`fold` is
per-event), not on test.

In [ ]:
# =============================== BACKEND ====================================
# Two GBDT backends behind one interface. sklearn's HistGradientBoosting was the
# only option before the vtx venv; xgboost is now the default for three reasons:
#
#   1. COLUMN SUBSAMPLING -- the real one. The per-sample ablation (§9b) showed
#      extreme redundancy: no feature block is worth >0.5 pts, and dropping all
#      11 jet-association features costs 0.1. colsample_by* is the standard
#      regularizer for precisely that regime, and sklearn's HistGradientBoosting
#      has no column-subsampling parameter at all.
#   2. Speed: hist + n_jobs=-1, which matters now that §9b fits 5+ models.
#   3. One library for these pointwise models and §12's ranker.
#
# Runs 1-4 were all sklearn. Set BACKEND="sklearn" to reproduce them exactly.
BACKEND = "xgb" if HAVE_XGB else "sklearn"

COMMON = dict(max_iter=2000, learning_rate=0.06, max_leaf_nodes=31,
              min_samples_leaf=50, l2_regularization=1.0,
              early_stopping=True, validation_fraction=0.15,
              random_state=RANDOM)

# NOTE min_child_weight is NOT min_samples_leaf. It is a sum of hessians, and for
# logloss the per-row hessian is p(1-p) <= 0.25, so 50 would mean 200+ rows per
# leaf -- far stricter than sklearn's 50. 5 is the closer analogue.
XGB_COMMON = dict(n_estimators=2000, learning_rate=0.06,
                  max_leaves=31, max_depth=0, grow_policy="lossguide",
                  min_child_weight=5, reg_lambda=1.0,
                  subsample=0.8, colsample_bytree=0.8,
                  tree_method="hist", n_jobs=-1, random_state=RANDOM,
                  early_stopping_rounds=50,
                  device=("cuda" if GPU else "cpu"))

# Early stopping validates on an EVENT-DISJOINT slice carved out of train.
# sklearn's validation_fraction splits ROWS at random, so two clusters of the same
# event can straddle its internal split; `fold` is per-event, so this cannot.
# Affects only where training stops, never the reported test metric -- but free.
VAL_LO = 0.60
def split_tv(frame):
    return frame[frame.fold < VAL_LO], frame[frame.fold >= VAL_LO]

# Optional: upweight the smaller samples so zjets (8.6% of events but where 26 of
# the ~30 available points live) is not drowned by VBF. The transfer matrix says
# the gain is ~0.2, so off by default.
BALANCE_SAMPLES = False
def sample_weights(frame):
    if not BALANCE_SAMPLES:
        return None
    n = frame.groupby("sample_id")["sample_id"].transform("size")
    return (len(frame) / (frame["sample_id"].nunique() * n)).to_numpy()

SK = (HistGradientBoostingClassifier, HistGradientBoostingRegressor)

def gbdt(task="clf", backend=None, **over):
    """Unfitted estimator for the active backend. task: 'clf' | 'reg'."""
    b = backend or BACKEND
    if b == "xgb":
        return (xgb.XGBClassifier if task == "clf" else xgb.XGBRegressor)(
            **{**XGB_COMMON, **over})
    return (HistGradientBoostingClassifier if task == "clf"
            else HistGradientBoostingRegressor)(**{**COMMON, **over})

def _target(frame, y):
    return y(frame) if callable(y) else frame[y]

def fit_gbdt(m, cols, y, frame, weights=True):
    """Fit on `frame`, early-stopping on its event-disjoint validation slice.
    `y` is a column name or a callable frame -> target."""
    trn, val = split_tv(frame)
    w = sample_weights(trn) if weights else None
    if isinstance(m, SK):
        m.fit(trn[cols], _target(trn, y), sample_weight=w)
    else:
        m.fit(trn[cols], _target(trn, y), sample_weight=w,
              eval_set=[(val[cols], _target(val, y))], verbose=False)
    return m

def proba(m, X):
    return m.predict_proba(X)[:, 1]

def n_trees(m):
    return getattr(m, "n_iter_", None) if isinstance(m, SK) else m.best_iteration

# ============================ REFERENCE MODEL ===============================
mB = fit_gbdt(gbdt("clf"), feat, "within60", train)
test["scoreB"] = proba(mB, test[feat])
print(f"backend: {BACKEND}   B within60 trees used: {n_trees(mB)}")
for s in sorted(test["sample_id"].unique()):
    sub = test[test.sample_id == s]
    print(f"  {SAMPLE_NAME.get(s,s):8s} TRKPTZ {select_metric(sub,'trkptz_score')[0]:5.1f}%"
          f"   B {select_metric(sub,'scoreB')[0]:5.1f}%"
          f"   oracle {select_metric(sub,'abs_dt',False)[0]:5.1f}%")

## 7. Track-tree plumbing and the per-track ceiling

Defines the streaming access used by §8, and computes the one number that bounds every
per-track method: selecting by `Σ pT·truth_is_hs`, i.e. a **perfect** HS tagger pooled the
simplest way. On the tight selection that was 80.2% on Z+jets against a 92.2% oracle —
the gap between them is *time quality*, not track identity, and no per-track weighting
can close it.

Worth recomputing here because the loose selection changes the sample.

In [ ]:
TRACK_LABEL    = "truth_is_hs"
TRACK_FEATURES = ["pt", "eta", "theta", "z0", "d0", "qOverP",
                  "sigma_z0", "sigma_d0", "sigma_qOverP",
                  "time", "timeRes", "time_valid", "quality", "nhgtd_hits",
                  "z0_pull_pv", "t_pull_cluster",
                  "dr_nearest_fwdjet", "pt_nearest_fwdjet", "is_ghost_of_nearest",
                  "is_lepton", "cluster_time", "cluster_delta_z"]
KEY  = ["sample_id", "event_num", "cluster_idx"]
READ = sorted(set(TRACK_FEATURES + KEY + [TRACK_LABEL]))
STEP = "300 MB"

# Truth can never be a feature; TRACK_FEATURES is an allowlist, but assert it anyway.
_bad = [f for f in TRACK_FEATURES if f.startswith("truth_") or f in set(TRUTH_ALIASES)]
assert not _bad, f"TRUTH IN TRACK FEATURES: {_bad}"

SAMPLE_PATHS    = {name: f"{p}:tracks" for name, p in FILES.items()}
ALL_TRACK_PATHS = list(SAMPLE_PATHS.values())
fold = ev.set_index(EVT)["fold"]          # same event-level split as the cluster model

def batch_fold(b):
    return fold.reindex(pd.MultiIndex.from_arrays(
        [b["sample_id"], b["event_num"]])).to_numpy()

# Batch-safe: a cluster's tracks can straddle a streaming batch boundary, so accumulate
# SUMS only and combine afterwards -- never a per-batch mean or rank.
parts = []
for b in uproot.iterate(ALL_TRACK_PATHS, READ, library="pd", step_size=STEP):
    bb = b.assign(_hs=b["pt"].to_numpy() * b[TRACK_LABEL].to_numpy())
    parts.append(bb.groupby(KEY, sort=False).agg(pt_hs=("_hs", "sum")))
agg_or = pd.concat(parts).groupby(level=[0, 1, 2]).agg(pt_hs=("pt_hs", "sum")).reset_index()
del parts
agg_or = agg_or.rename(columns={"pt_hs": "trk_oracle"})
test = test.merge(agg_or, on=KEY, how="left")
print(f"aggregated {len(agg_or):,} clusters\n")

print(f"{'selector':32s}" + "".join(f"{SAMPLE_NAME.get(s,s):>10s}"
                                    for s in sorted(test['sample_id'].unique())))
for nm, col, hib in [("TRKPTZ (baseline)", "trkptz_score", True),
                     ("cluster model B", "scoreB", True),
                     ("Sum pT*truth [TRACK ORACLE]", "trk_oracle", True),
                     ("cluster oracle (ceiling)", "abs_dt", False)]:
    print(f"{nm:32s}" + "".join(
        f"{select_metric(test[test.sample_id == s], col, hib)[0]:9.1f}%"
        for s in sorted(test["sample_id"].unique())))

## 8. Deep Sets, end-to-end on the selection objective (+ auxiliary head)

`cluster_score = ρ( POOL_tracks φ(track) )`, trained directly on *which cluster to pick*
— a multi-positive listwise softmax over the event's clusters, where every cluster inside
the 60 ps window counts as correct. That makes the loss the physics metric itself rather
than a proxy, which is the only change in this entire study that ever moved Z+jets (+3.5).

**What's new: an auxiliary per-track head.** φ gets a second output trained on
`truth_is_hs`, added to the loss with weight λ:

```
loss = listwise_CE(selection)  +  λ · BCE(φ_aux(track), truth_is_hs)
```

The motivation is that the listwise loss is *very* sparse — one bit per event — and a
dense per-track signal may shape φ better.

> **Why λ is scanned rather than set.** Adding information has repeatedly *hurt* in this
> study: §15's context features raised AUC and lowered core fraction; run 2's 5.7×
> capacity lost 0.2 everywhere; dropping the jet block *helped* Z+jets. And there is a
> specific risk here — §14 showed a perfect `truth_is_hs` tagger caps at 80.2%, so it is a
> **demonstrably imperfect** target, and pulling φ toward it may degrade the
> representation. Scanning λ **including 0** makes the control internal: λ=0 reproduces
> the current model exactly, so the worst case is "λ=0 wins" and we lose only compute.

Model selection is on a validation slice carved from **train** (`DS_VAL_LO`), never on
test, using the macro-mean across samples so VBF cannot dominate the choice.

In [ ]:
# ===== §17. Phase 4 — Deep Sets over tracks, trained on the SELECTION objective =
# cluster_score = rho( POOL_tracks phi(track) ), trained directly on "which cluster
# should be picked". phi replaces the hand-fixed pT weight, POOL/rho replace the
# fixed sum-then-compare. Neither is bounded by the 80.2% that Sum pT*truth caps at.
import torch, torch.nn as nn, torch.nn.functional as F, time, copy

# Everything below runs on DEV. The per-sample track tensors are a few hundred
# MB, so they live on the device for the whole run rather than being copied per
# batch; only the small index arrays cross the boundary each step.
DEV = torch.device('cuda' if GPU else 'cpu')
print(f'device: {DEV}')

DS_TRAIN_EVENTS = 150_000     # split PER SAMPLE (see quota block)
DS_TEST_EVENTS  = 100_000
DS_EPOCHS, DS_BATCH_EV, DS_LR = 15, 512, 2e-3
# Run 2 raised these to 256/128 (138k params) with cosine decay and 3x zjets
# weighting: vbf 93.4 / zjets 67.9 / dijet 90.0, i.e. ~0.2 WORSE than run 1 on all
# three, and the weighting did not even help zjets. Capacity is not the binding
# constraint, so back to the smaller, 1.8x cheaper model with equal event weights.
DS_HIDDEN, DS_EMBED = 96, 64
DS_SAMPLE_WEIGHTS = {}               # {"zjets": 3.0} to re-try emphasis
DS_VAL_LO = 0.60                     # train-fold events >= this are VALIDATION
DS_POOLS = ["sum"]                   # gate matched sum to 0.1 -- see archive
# Auxiliary per-track head weight. SCANNED, not set: adding information has hurt
# repeatedly in this study, and truth_is_hs is a demonstrably imperfect target (a
# perfect tagger caps at 80.2% vs a 92.2% oracle). lam=0 reproduces the plain model
# exactly, so it is the internal control and the worst case is "0 wins".
DS_AUX_LAMBDAS = [0.0, 0.1, 0.3, 1.0]
DS_FEATURES = list(TRACK_FEATURES)
torch.manual_seed(RANDOM)
rng_ds = np.random.default_rng(RANDOM)

# ---- bounded subset of events, with PER-SAMPLE quotas -----------------------
# A global cap filled sequentially reads vbf first (620k events) and never reaches
# zjets or dijet, silently training on one topology. That bug has appeared three
# times (S11 pass 1, S14's AUC table, the first version of this cell), so the quota
# is per sample and the composition is asserted below.
DS_TRAIN_PER = DS_TRAIN_EVENTS // max(1, len(SAMPLE_PATHS))
DS_TEST_PER  = DS_TEST_EVENTS  // max(1, len(SAMPLE_PATHS))
keep_tr, keep_te = [], []
print(f"per-sample quota: {DS_TRAIN_PER:,} train / {DS_TEST_PER:,} test events")
for _name, _path in SAMPLE_PATHS.items():
    ntr = nte = 0
    for b in uproot.iterate(_path, READ, library="pd", step_size=STEP):
        b = b.assign(fold=batch_fold(b))
        for is_tr in (True, False):
            cap = DS_TRAIN_PER if is_tr else DS_TEST_PER
            got = ntr if is_tr else nte
            if got >= cap:
                continue
            sel = b[(b.fold < 0.7) if is_tr else (b.fold >= 0.7)]
            if not len(sel):
                continue
            ev_ = sel[EVT].drop_duplicates().iloc[:max(0, cap - got)]
            sel = sel.merge(ev_, on=EVT, how="inner")
            (keep_tr if is_tr else keep_te).append(sel)
            if is_tr: ntr += len(ev_)
            else:     nte += len(ev_)
        if ntr >= DS_TRAIN_PER and nte >= DS_TEST_PER:
            break
    print(f"  {_name:6s} train {ntr:>7,} ev   test {nte:>7,} ev"
          + ("" if ntr >= DS_TRAIN_PER else "   (all available)"))
TR_T = pd.concat(keep_tr, ignore_index=True); TE_T = pd.concat(keep_te, ignore_index=True)
del keep_tr, keep_te
assert TR_T["sample_id"].nunique() == len(SAMPLE_PATHS), \
    f"only {TR_T['sample_id'].nunique()} of {len(SAMPLE_PATHS)} samples in train"

# Model selection needs its own split. Picking the best epoch on TEST would be
# test-set peeking; `fold` is per-event so this validation slice is event-disjoint.
FIT_T = TR_T[TR_T.fold <  DS_VAL_LO]
VAL_T = TR_T[TR_T.fold >= DS_VAL_LO]
print(f"tracks: fit {len(FIT_T):,} rows / {FIT_T.groupby(EVT).ngroups:,} ev   "
      f"val {len(VAL_T):,} / {VAL_T.groupby(EVT).ngroups:,} ev   "
      f"test {len(TE_T):,} / {TE_T.groupby(EVT).ngroups:,} ev")

# ---- standardise on FIT only; keep explicit missing-indicators ---------------
# NaN means "no valid time" and 0.0 is a legitimate time, so: standardise, fill
# with 0 (= the mean), and hand the net a flag saying it was missing.
MU = FIT_T[DS_FEATURES].mean()
SD = FIT_T[DS_FEATURES].std().replace(0, 1.0)
NA_COLS = [c for c in DS_FEATURES if FIT_T[c].isna().any()]
print(f"{len(DS_FEATURES)} features + {len(NA_COLS)} missing-indicators")

def prep(frame):
    """Sort by (event, cluster) so tracks->clusters->events are all contiguous."""
    f = frame.sort_values(KEY, kind="mergesort").reset_index(drop=True)
    X = ((f[DS_FEATURES] - MU) / SD)
    M = f[NA_COLS].isna().astype(np.float32).to_numpy()
    X = np.nan_to_num(X.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    X = np.hstack([X, M]) if len(NA_COLS) else X
    cl_id, cl_key = pd.factorize(pd.MultiIndex.from_frame(f[KEY]), sort=False)
    cl = pd.DataFrame(list(cl_key), columns=KEY)
    ev_id, _ = pd.factorize(pd.MultiIndex.from_frame(cl[EVT]), sort=False)
    lab = df[KEY + ["within60"]].drop_duplicates(KEY)      # sample_id is in KEY
    m = cl.merge(lab, on=KEY, how="left")
    # per-TRACK auxiliary target, aligned with X because f is sorted by KEY
    a = torch.from_numpy(f[TRACK_LABEL].to_numpy(np.float32)).to(DEV)
    return dict(X=torch.from_numpy(X).to(DEV),
                c=torch.from_numpy(cl_id.astype(np.int64)).to(DEV),
                e=torch.from_numpy(ev_id.astype(np.int64)).to(DEV),
                y=torch.from_numpy(m["within60"].fillna(0)
                                    .to_numpy(np.float32)).to(DEV),
                a=a,
                m=m, ncl=len(m), nev=int(ev_id.max()) + 1)

FIT, VAL, TST = prep(FIT_T), prep(VAL_T), prep(TE_T)

def event_weights(d):
    w = np.ones(d["nev"], np.float32)
    if DS_SAMPLE_WEIGHTS:
        per_cl = d["m"]["sample_id"].map(
            lambda s: DS_SAMPLE_WEIGHTS.get(SAMPLE_NAME.get(s, s), 1.0)).to_numpy(np.float32)
        w[d["e"].numpy()] = per_cl
        w /= max(w.mean(), 1e-9)
    return torch.from_numpy(w).to(DEV)
Wfit = event_weights(FIT)

# ---- model -------------------------------------------------------------------
class DeepSets(nn.Module):
    """pool='sum'  : pooled = SUM_i phi(x_i)            -- every track adds.
       pool='gate' : pooled = SUM_i sigmoid(a(x_i)) phi(x_i)
                     A learned per-track GATE, i.e. an unnormalised attention. It
                     lets a few decisive tracks dominate and lets the net suppress
                     pileup outright, while KEEPING cluster magnitude -- which
                     softmax attention would destroy, since normalising weights to
                     sum to 1 makes a 20-track and a 2-track cluster identical and
                     Sum pT is one of the strongest signals we have."""
    def __init__(self, d_in, d_h=DS_HIDDEN, d_e=DS_EMBED, pool="sum", aux=False):
        super().__init__()
        self.pool, self.use_aux = pool, aux
        self.phi = nn.Sequential(nn.Linear(d_in, d_h), nn.ReLU(),
                                 nn.Linear(d_h, d_h), nn.ReLU(), nn.Linear(d_h, d_e))
        if pool == "gate":
            self.att = nn.Sequential(nn.Linear(d_in, d_h), nn.ReLU(), nn.Linear(d_h, 1))
        self.rho = nn.Sequential(nn.Linear(d_e, d_h), nn.ReLU(), nn.Linear(d_h, 1))
        # Auxiliary head reads phi's OWN embedding, so the gradient it sends lands on
        # the shared per-track representation -- that is the whole point.
        if aux:
            self.aux = nn.Linear(d_e, 1)
    def forward(self, x, cidx, ncl, want_aux=False):
        h = self.phi(x)
        a = self.aux(h).squeeze(-1) if (want_aux and self.use_aux) else None
        if self.pool == "gate":
            h = h * torch.sigmoid(self.att(x))
        pooled = torch.zeros(ncl, h.shape[1], dtype=h.dtype,
                             device=h.device).index_add_(0, cidx, h)
        sc = self.rho(pooled).squeeze(-1)
        return (sc, a) if want_aux else sc

def listwise_ce(scores, eidx, pos, nev, w=None):
    """-log( softmax mass on ACCEPTABLE clusters ), per event, optionally weighted.
    Multi-positive: any cluster inside the window is a correct pick, which IS the
    physics metric rather than a proxy for it."""
    m = torch.full((nev,), -1e30, dtype=scores.dtype,
                   device=scores.device).scatter_reduce(
        0, eidx, scores, reduce="amax", include_self=True)
    e  = torch.exp(scores - m[eidx])
    Z  = torch.zeros(nev, dtype=scores.dtype, device=scores.device)\
           .index_add_(0, eidx, e)
    P  = torch.zeros(nev, dtype=scores.dtype, device=scores.device)\
           .index_add_(0, eidx, e * pos)
    ok = P > 0
    l  = -(torch.log(P[ok] + 1e-12) - torch.log(Z[ok] + 1e-12))
    if w is None:
        return l.mean()
    ww = w[ok]
    return (l * ww).sum() / ww.sum().clamp_min(1e-9)

def spans(d):
    c = d["c"].cpu().numpy(); e = d["e"].cpu().numpy()
    return (np.searchsorted(c, np.arange(d["ncl"])),
            np.searchsorted(c, np.arange(d["ncl"]), side="right"),
            np.searchsorted(e, np.arange(d["nev"])),
            np.searchsorted(e, np.arange(d["nev"]), side="right"))
CS, CE_, ES, EE = spans(FIT)

def evaluate(net, d):
    net.eval()
    with torch.no_grad():
        s = net(d["X"], d["c"], d["ncl"]).cpu().numpy()
    net.train()
    out = d["m"].copy(); out["ds"] = s
    per = []
    for sid in sorted(out["sample_id"].dropna().unique()):
        sub = out[out.sample_id == sid]
        pick = sub.loc[sub.groupby(EVT, sort=False)["ds"].idxmax()]
        per.append((SAMPLE_NAME.get(sid, sid), 100 * pick["within60"].mean()))
    pick = out.loc[out.groupby(EVT, sort=False)["ds"].idxmax()]
    return per, 100 * pick["within60"].mean(), out

def train(pool, lam=0.0):
    torch.manual_seed(RANDOM)
    net = DeepSets(FIT["X"].shape[1], pool=pool, aux=lam > 0).to(DEV)
    opt = torch.optim.Adam(net.parameters(), lr=DS_LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=DS_EPOCHS,
                                                       eta_min=DS_LR / 20)
    tag = f"{pool} lam={lam:g}"
    print(f"\n[{tag}] parameters: {sum(p.numel() for p in net.parameters()):,}")
    order = np.arange(FIT["nev"])
    best, best_state, best_ep = -1.0, None, -1
    for ep in range(DS_EPOCHS):
        rng_ds.shuffle(order); tot, nb, t0 = 0.0, 0, time.time()
        for i in range(0, len(order), DS_BATCH_EV):
            evs = order[i:i + DS_BATCH_EV]
            cl_idx = np.concatenate([np.arange(ES[e], EE[e]) for e in evs])
            tk_idx = np.concatenate([np.arange(CS[c], CE_[c]) for c in cl_idx])
            loc_c  = np.repeat(np.arange(len(cl_idx)), CE_[cl_idx] - CS[cl_idx])
            loc_e  = np.repeat(np.arange(len(evs)), EE[evs] - ES[evs])
            tk_t = torch.from_numpy(tk_idx).to(DEV)
            cl_t = torch.from_numpy(cl_idx).to(DEV)
            # want_aux=True ALWAYS, even at lam=0. forward() returns a bare
            # tensor when want_aux is False, so `sc, aux = <tensor>` would raise
            # ValueError (too many values to unpack) and the lam=0 control -- the
            # baseline the whole scan is measured against -- would never run.
            # With want_aux=True the model returns (sc, None) when it has no aux
            # head, and the `lam > 0` guard below leaves the loss unchanged.
            sc, aux = net(FIT["X"][tk_t], torch.from_numpy(loc_c).to(DEV),
                          len(cl_idx), want_aux=True)
            loss = listwise_ce(sc, torch.from_numpy(loc_e).to(DEV),
                               FIT["y"][cl_t], len(evs),
                               Wfit[torch.from_numpy(evs).to(DEV)])
            if lam > 0:
                loss = loss + lam * F.binary_cross_entropy_with_logits(
                    aux, FIT["a"][tk_t])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss); nb += 1
        sched.step()
        vper, _, _ = evaluate(net, VAL)
        macro = float(np.mean([v for _, v in vper]))   # macro so vbf cannot dominate
        star = ""
        if macro > best:
            best, best_state, best_ep, star = macro, copy.deepcopy(net.state_dict()), ep + 1, "  *"
        print(f"  [{tag}] epoch {ep+1:>2}/{DS_EPOCHS}  loss {tot/max(nb,1):.4f}  "
              f"[{time.time()-t0:.0f}s]  VAL "
              + "  ".join(f"{n} {v:.1f}%" for n, v in vper) + f"  macro {macro:.2f}{star}")
    net.load_state_dict(best_state)
    print(f"  [{tag}] restored epoch {best_ep} (VAL macro {best:.2f})")
    return net, best

RESULTS, VALMACRO = {}, {}
for _p in DS_POOLS:
    for _lam in DS_AUX_LAMBDAS:
        _net, _v = train(_p, _lam)
        _k = f"{_p} lam={_lam:g}"
        RESULTS[_k], VALMACRO[_k] = evaluate(_net, TST), _v

# ---- final comparison on the SAME event subset -------------------------------
_first = next(iter(RESULTS))
per0 = RESULTS[_first][0]
ref = RESULTS[_first][2].merge(
    df[KEY + ["trkptz_score"]].drop_duplicates(KEY), on=KEY, how="left")
if "T3" in globals():
    ref = ref.merge(T3[KEY + [c for c in ("raw base (22 local)", "trk_oracle")
                              if c in T3.columns]], on=KEY, how="left")
print(f"\n{'selector':30s}" + "".join(f"{n:>10s}" for n, _ in per0) + f"{'ALL':>10s}")
for nm, col in [("TRKPTZ", "trkptz_score"), ("Sum pT*P(HS)", "raw base (22 local)"),
                ("Sum pT*truth [track oracle]", "trk_oracle")]:
    if col not in ref.columns:
        continue
    row = []
    for n, _ in per0:
        sid = {v: k for k, v in SAMPLE_NAME.items()}[n]
        sub = ref[ref.sample_id == sid]
        row.append(100 * sub.loc[sub.groupby(EVT, sort=False)[col].idxmax(), "within60"].mean())
    allp = ref.loc[ref.groupby(EVT, sort=False)[col].idxmax()]
    print(f"{nm:30s}" + "".join(f"{v:9.1f}%" for v in row)
          + f"{100*allp['within60'].mean():9.1f}%")
for _k, (per, pooled, _) in RESULTS.items():
    print(f"{'Deep Sets [' + _k + ']':30s}" + "".join(f"{v:9.1f}%" for _, v in per)
          + f"{pooled:9.1f}%")

# The honest pick: best VAL macro, chosen without looking at test.
_best = max(VALMACRO, key=VALMACRO.get)
print(f"\nselected on VALIDATION: {_best}  (val macro {VALMACRO[_best]:.2f})")
print("lambda scan, val macro:  "
      + "   ".join(f"{k.split('lam=')[1]}: {v:.2f}" for k, v in VALMACRO.items()))
if not _best.endswith("lam=0"):
    print("  -> the auxiliary head helped; truth_is_hs shapes phi usefully")
else:
    print("  -> lambda=0 won; the auxiliary target does not improve the "
          "representation (consistent with its 80.2% ceiling)")
print("\nNote: evaluated on the DS event subset with BALANCED per-sample quotas, so")
print("absolute values -- especially ALL -- differ from the §7/§15 tables. Compare")
print("the rows above against each other, not across cells.")